In [1]:
# Environment setup

import warnings, os, sys
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import importlib.metadata
try:
    import pkg_resources
except ImportError:
    import types
    pkg_resources = types.ModuleType("pkg_resources")
    pkg_resources.get_distribution = lambda n: type("D", (), {"version": importlib.metadata.version(n)})()
    sys.modules["pkg_resources"] = pkg_resources

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso, ElasticNet, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

In [2]:
#Configuration

PATH_CLEANED   = "listings_cleaned_keer.csv"
PATH_GENDER    = "202410_listings_with_gender_Cong.csv"
PATH_POI       = "202410_listings_with_POIbuffer_Cong.csv"
PATH_SENTIMENT = "airbnb_sentiment_results_ml.csv"

POI_COLS = ["Total_POI_500m", "Total_POI_800m", "Total_POI_1500m"]

OUTPUT_DIR = "./results_rc_script2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

HYPEROPT_EVALS_XGB = 300
HYPEROPT_EVALS_LGB = 200
EARLY_STOPPING_ROUNDS = 20
N_FOLDS = 5

TARGET = "log_review_count"

In [3]:
#Build experiment list

EXPERIMENTS = []
for i, poi_col in enumerate(POI_COLS, start=5):
    radius = poi_col.replace("Total_POI_", "")
    EXPERIMENTS.append({"name": f"M_{i}_POI{radius}", "poi": poi_col})

print(f"Total experiments: {len(EXPERIMENTS)}")
for e in EXPERIMENTS:
    print(f"  {e['name']:<30}  POI={e['poi']}")

master_summary = []

Total experiments: 3
  M_5_POI500m                     POI=Total_POI_500m
  M_6_POI800m                     POI=Total_POI_800m
  M_7_POI1500m                    POI=Total_POI_1500m


In [4]:
#Helper: metrics

def compute_metrics(y_true, y_pred_log, n_features):
    y_true_orig = np.expm1(np.asarray(y_true))
    y_pred_orig = np.expm1(np.asarray(y_pred_log))
    errors = y_true_orig - y_pred_orig
    n = len(y_true)
    mae  = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    mape = np.mean(np.abs(errors / np.where(y_true_orig == 0, 1e-10, y_true_orig))) * 100
    r2   = r2_score(y_true_orig, y_pred_orig)
    adj  = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    rlog = np.sqrt(mean_squared_error(y_true, y_pred_log))
    r2_log = r2_score(np.asarray(y_true), np.asarray(y_pred_log))
    return dict(MAE=mae, RMSE=rmse, RMSE_log=rlog, MAPE=mape,
                R2=r2, Adj_R2=adj, R2_log=r2_log)

In [5]:
#Helper: XGBoost hyperopt

def tune_xgb(X_tr, y_tr, X_va, y_va):
    space = {
        "gamma":            hp.uniform("gamma", 0.0, 0.5),
        "reg_alpha":        hp.uniform("reg_alpha", 0.0, 2.0),
        "reg_lambda":       hp.uniform("reg_lambda", 0.0, 2.0),
        "max_depth":        hp.quniform("max_depth", 3, 8, 1),
        "n_estimators":     hp.quniform("n_estimators", 400, 800, 1),
        "colsample_bytree": hp.uniform("colsample_bytree", 0.4, 1.0),
        "min_child_weight": hp.uniform("min_child_weight", 1, 10),
        "learning_rate":    hp.uniform("learning_rate", 0.01, 0.3),
        "subsample":        hp.uniform("subsample", 0.6, 1.0),
    }
    def objective(p):
        m = xgb.XGBRegressor(
            n_estimators=int(p["n_estimators"]), max_depth=int(p["max_depth"]),
            gamma=p["gamma"], reg_alpha=p["reg_alpha"], reg_lambda=p["reg_lambda"],
            colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
            learning_rate=p["learning_rate"], subsample=p["subsample"],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            tree_method="hist", eval_metric="rmse",
            n_jobs=-1, random_state=42, verbosity=0,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return {"loss": np.sqrt(mean_squared_error(y_va, m.predict(X_va))), "status": STATUS_OK}
    best = fmin(objective, space, algo=tpe.suggest, max_evals=HYPEROPT_EVALS_XGB,
                trials=Trials(), rstate=np.random.default_rng(42), verbose=False)
    bp = {
        "gamma": best["gamma"], "reg_alpha": best["reg_alpha"], "reg_lambda": best["reg_lambda"],
        "colsample_bytree": best["colsample_bytree"], "min_child_weight": best["min_child_weight"],
        "learning_rate": best["learning_rate"], "subsample": best["subsample"],
        "max_depth": int(best["max_depth"]), "n_estimators": int(best["n_estimators"]),
    }
    final = xgb.XGBRegressor(**bp, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                             tree_method="hist", eval_metric="rmse",
                             n_jobs=-1, random_state=42, verbosity=0)
    final.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return final, bp

In [6]:
#Helper: LightGBM hyperopt

def tune_lgb(X_tr, y_tr, X_va, y_va):
    space = {
        "n_estimators":     hp.quniform("n_estimators", 300, 1200, 1),
        "num_leaves":       hp.quniform("num_leaves", 10, 80, 1),
        "learning_rate":    hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)),
        "feature_fraction": hp.uniform("feature_fraction", 0.4, 1.0),
        "bagging_fraction": hp.uniform("bagging_fraction", 0.4, 1.0),
        "bagging_freq":     hp.quniform("bagging_freq", 1, 10, 1),
        "min_data_in_leaf": hp.quniform("min_data_in_leaf", 5, 50, 1),
        "reg_alpha":        hp.loguniform("reg_alpha", np.log(1e-5), np.log(2.0)),
        "reg_lambda":       hp.loguniform("reg_lambda", np.log(1e-5), np.log(2.0)),
    }
    def objective(p):
        m = lgb.LGBMRegressor(
            n_estimators=int(p["n_estimators"]), num_leaves=int(p["num_leaves"]),
            learning_rate=p["learning_rate"], feature_fraction=p["feature_fraction"],
            bagging_fraction=p["bagging_fraction"], bagging_freq=int(p["bagging_freq"]),
            min_data_in_leaf=int(p["min_data_in_leaf"]),
            reg_alpha=p["reg_alpha"], reg_lambda=p["reg_lambda"],
            objective="regression", random_state=9, verbose=-1, n_jobs=-1,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)])
        return {"loss": np.sqrt(mean_squared_error(y_va, m.predict(X_va))), "status": STATUS_OK}
    best = fmin(objective, space, algo=tpe.suggest, max_evals=HYPEROPT_EVALS_LGB,
                trials=Trials(), rstate=np.random.default_rng(9), verbose=False)
    bp = {
        "n_estimators": int(best["n_estimators"]), "num_leaves": int(best["num_leaves"]),
        "learning_rate": best["learning_rate"], "feature_fraction": best["feature_fraction"],
        "bagging_fraction": best["bagging_fraction"], "bagging_freq": int(best["bagging_freq"]),
        "min_data_in_leaf": int(best["min_data_in_leaf"]),
        "reg_alpha": best["reg_alpha"], "reg_lambda": best["reg_lambda"],
    }
    final = lgb.LGBMRegressor(**bp, objective="regression", random_state=9, verbose=-1, n_jobs=-1)
    final.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)])
    return final, bp

In [7]:
#Helper: Averaging ensemble class

class AveragingModels(BaseEstimator, RegressorMixin, TransformerMixin):
    def __init__(self, models):
        self.models = models
    def fit(self, X, y):
        self.models_ = [clone(m) for m in self.models]
        for m in self.models_: m.fit(X, y)
        return self
    def predict(self, X):
        return np.mean(np.column_stack([m.predict(X) for m in self.models_]), axis=1)


In [8]:
#Main loop

for idx, exp in enumerate(EXPERIMENTS, start=1):
    RUN_LABEL = exp["name"]
    print("\n" + "=" * 72)
    print(f"  [{idx}/{len(EXPERIMENTS)}] {RUN_LABEL}   POI={exp['poi']}")
    print("=" * 72)

    cleaned   = pd.read_csv(PATH_CLEANED)
    gender    = pd.read_csv(PATH_GENDER)[["id", "gender_final"]]
    poi_df    = pd.read_csv(PATH_POI)[["id", "Total_POI_500m", "Total_POI_800m", "Total_POI_1500m"]]
    sentiment = (pd.read_csv(PATH_SENTIMENT)
                   .rename(columns={"listing_id": "id"})
                   [["id", "average_sentiment", "review_count"]])

    for frame in [cleaned, gender, poi_df, sentiment]:
        frame["id"] = frame["id"].astype(str)

    if "room_type" in cleaned.columns and "room_type_code" not in cleaned.columns:
        cleaned["room_type"] = cleaned["room_type"].astype(str)
        cleaned["room_type_code"] = LabelEncoder().fit_transform(cleaned["room_type"])

    df = (cleaned
          .merge(gender, on="id", how="left")
          .merge(poi_df, on="id", how="left")
          .merge(sentiment, on="id", how="left"))

    df["log_review_count"] = np.log1p(df["review_count"])

    p95 = df["review_count"].quantile(0.95)
    df = df[(df["review_count"] <= p95) | (df["review_count"].isna())].copy()
    print(f"  After 95-percentile trim: {len(df)} rows")

    multi_listing = df["calculated_host_listings_count"] >= 2
    entire_home = (df["room_type"].str.contains("Entire", case=False, na=False)
                   if "room_type" in df.columns else pd.Series(True, index=df.index))
    df["professional_host"] = (multi_listing & entire_home).astype(int)

    # CORE_IVS = only the selected POI radius
    CORE_IVS = [exp["poi"]]

    CONTROLS_STRUCTURAL = ["room_type_code", "accommodates", "bathrooms", "bedrooms", "beds",
                           "property_Private room", "property_Shared room", "property_Unique stays",
                           "professional_host"]
    CONTROLS_HOST = ["host_since", "host_response_time_code", "host_response_rate",
                     "host_acceptance_rate", "host_is_superhost", "host_has_profile_pic",
                     "host_identity_verified", "host_listings_count",
                     "calculated_host_listings_count", "gender_final"]
    CONTROLS_POLICY = ["minimum_nights", "maximum_nights", "availability_30",
                       "instant_bookable", "has_availability", "inactive"]
    CONTROLS_SENTIMENT = ["average_sentiment", "log_price"]
    CONTROLS_AMENITY = ["amenity_tv", "amenity_netflix", "amenity_gym", "amenity_elevator",
                        "amenity_fridge", "amenity_heating", "amenity_hair_dryer",
                        "amenity_air_conditioning", "amenity_hot_tub", "amenity_oven",
                        "amenity_bbq", "amenity_security_cameras", "amenity_workspace",
                        "amenity_coffee", "amenity_backyard", "amenity_outdoor_dining",
                        "amenity_greets", "amenity_pool", "amenity_beachfront",
                        "amenity_patio", "amenity_luggage", "amenity_furniture"]

    ALL_FEATURES_REQUESTED = (CORE_IVS + CONTROLS_STRUCTURAL + CONTROLS_HOST +
                              CONTROLS_POLICY + CONTROLS_SENTIMENT + CONTROLS_AMENITY)
    ALL_FEATURES = [c for c in ALL_FEATURES_REQUESTED if c in df.columns]

    BINARY_COLS = [c for c in ALL_FEATURES
                   if c.startswith("amenity_") or c in {
                       "host_is_superhost", "host_has_profile_pic",
                       "host_identity_verified", "instant_bookable",
                       "has_availability", "inactive", "gender_final",
                       "professional_host"}]
    CONTINUOUS_COLS = [c for c in ALL_FEATURES if c not in BINARY_COLS]

    data = df[ALL_FEATURES + [TARGET]].copy().dropna(subset=[TARGET])
    X = data[ALL_FEATURES].copy()
    y = data[TARGET].copy()

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=42)
    X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

    for col in ALL_FEATURES:
        for split in [X_train, X_val, X_test]:
            split[col] = pd.to_numeric(split[col], errors="coerce")
    X_train = X_train.dropna(axis=1, how="all")
    X_val   = X_val[X_train.columns]
    X_test  = X_test[X_train.columns]
    ALL_FEATURES    = list(X_train.columns)
    CORE_IVS        = [c for c in CORE_IVS        if c in ALL_FEATURES]
    BINARY_COLS     = [c for c in BINARY_COLS     if c in ALL_FEATURES]
    CONTINUOUS_COLS = [c for c in CONTINUOUS_COLS if c in ALL_FEATURES and c != "log_price"]

    for col in ["average_sentiment", "log_price"]:
        if col in ALL_FEATURES:
            m = X_train[col].median()
            for split in [X_train, X_val, X_test]:
                split[col] = split[col].fillna(m)

    for col in BINARY_COLS:
        mv = X_train[col].mode()
        if len(mv) == 0: continue
        for split in [X_train, X_val, X_test]:
            split[col] = split[col].fillna(mv[0])

    imp = KNNImputer(n_neighbors=5)
    imp.fit(X_train[CONTINUOUS_COLS])
    for split in [X_train, X_val, X_test]:
        split[CONTINUOUS_COLS] = imp.transform(split[CONTINUOUS_COLS])

    # log_price refined RF imputation
    if "log_price" in ALL_FEATURES:
        price_pred_cols = [c for c in CONTINUOUS_COLS if c != "log_price"]
        train_known   = X_train[X_train["log_price"].notna()].copy()
        train_missing = X_train[X_train["log_price"].isna()].copy()
        if len(train_missing) > 0:
            rf_price = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
            rf_price.fit(
                train_known[price_pred_cols].fillna(train_known[price_pred_cols].median()),
                train_known["log_price"]
            )
            X_train.loc[X_train["log_price"].isna(), "log_price"] = rf_price.predict(
                train_missing[price_pred_cols].fillna(train_known[price_pred_cols].median())
            )
            for split in [X_val, X_test]:
                missing_mask = split["log_price"].isna()
                if missing_mask.sum() > 0:
                    split.loc[missing_mask, "log_price"] = rf_price.predict(
                        split.loc[missing_mask, price_pred_cols].fillna(
                            train_known[price_pred_cols].median()
                        )
                    )

    for frame in [X_train, X_val, X_test, y_train, y_val, y_test]:
        frame.reset_index(drop=True, inplace=True)

    print("  Tuning Elastic Net ...")
    gs_enet = GridSearchCV(
        make_pipeline(RobustScaler(), ElasticNet(random_state=3)),
        {"elasticnet__alpha":    [1e-6, 1e-4, 1e-2, 0.1, 0.3, 0.5],
         "elasticnet__l1_ratio": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]},
        cv=5, scoring="neg_mean_squared_error", n_jobs=1
    )
    gs_enet.fit(X_train, y_train)

    print("  Tuning Random Forest ...")
    gs_rf = GridSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {"n_estimators": [100, 200, 300],
         "max_features": [0.3, 0.4, 0.5, "sqrt"],
         "min_samples_leaf": [2, 3, 5]},
        cv=5, scoring="neg_mean_squared_error", n_jobs=1
    )
    gs_rf.fit(X_train, y_train)

    print(f"  Tuning XGBoost ({HYPEROPT_EVALS_XGB} trials) ...")
    model_xgb, xgb_bp = tune_xgb(X_train.values, y_train.values, X_val.values, y_val.values)

    print(f"  Tuning LightGBM ({HYPEROPT_EVALS_LGB} trials) ...")
    model_lgb, lgb_bp = tune_lgb(X_train.values, y_train.values, X_val.values, y_val.values)

    lasso    = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=1))
    ridge    = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0001, l1_ratio=0.0, random_state=3))
    enet     = make_pipeline(RobustScaler(), ElasticNet(
                 alpha=gs_enet.best_params_["elasticnet__alpha"],
                 l1_ratio=gs_enet.best_params_["elasticnet__l1_ratio"], random_state=3))
    krr      = make_pipeline(RobustScaler(), KernelRidge(alpha=0.6, kernel="polynomial",
                                                         degree=2, coef0=2.5))
    bayridge = make_pipeline(RobustScaler(), BayesianRidge())
    gboost = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4,
                                       max_features="sqrt", min_samples_leaf=15,
                                       min_samples_split=10, loss="huber", random_state=5)
    rf = RandomForestRegressor(**gs_rf.best_params_, random_state=42, n_jobs=-1)
    model_xgb_plain = xgb.XGBRegressor(**xgb_bp, tree_method="hist", eval_metric="rmse",
                                       n_jobs=-1, random_state=42, verbosity=0)
    averaged = AveragingModels(models=(model_lgb, gboost, rf, lasso, enet, krr, model_xgb_plain))

    MODEL_DICT = {
        "Lasso": lasso, "Ridge": ridge, "Elastic Net": enet, "Kernel Ridge": krr,
        "Bayesian Ridge": bayridge, "Random Forest": rf, "Gradient Boost": gboost,
        "XGBoost": model_xgb, "XGBoost (plain)": model_xgb_plain, "LightGBM": model_lgb,
        "Averaged Ensemble": averaged,
    }

    print("  Fitting all models ...")
    for name, model in MODEL_DICT.items():
        if name == "XGBoost": continue
        try:    model.fit(X_train, y_train)
        except: model.fit(X_train.values, y_train)

    results = []
    all_preds = {}
    n_feat = X_test.shape[1]
    for name, model in MODEL_DICT.items():
        try:    yp = model.predict(X_test)
        except: yp = model.predict(X_test.values)
        all_preds[name] = yp
        m = compute_metrics(y_test, yp, n_feat)
        m["Model"] = name
        results.append(m)
    results_df = pd.DataFrame(results).sort_values("RMSE_log")

    print(f"\n  Best: {results_df.iloc[0]['Model']}  "
          f"RMSE_log={results_df.iloc[0]['RMSE_log']:.4f}  "
          f"R²={results_df.iloc[0]['R2']:.4f}")

    results_df.to_csv(f"{OUTPUT_DIR}/model_results_{RUN_LABEL}.csv", index=False)

    PALETTE = ["#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",
               "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#BAB0AC", "#D37295"]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, metric in zip(axes.flatten(), ["MAE", "RMSE", "RMSE_log", "MAPE", "R2", "Adj_R2"]):
        hb = metric in ("R2", "Adj_R2")
        ds = results_df.sort_values(metric, ascending=not hb)
        bars = ax.barh(ds["Model"], ds[metric], color=PALETTE[:len(ds)], alpha=0.85)
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} ({'higher' if hb else 'lower'} is better)", fontweight="bold")
        ax.invert_yaxis()
        for bar, v in zip(bars, ds[metric]):
            ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                    f"{v:.4f}", va="center", fontsize=9)
    plt.suptitle(f"Model Comparison — {RUN_LABEL}  (target=review_count)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/model_comparison_{RUN_LABEL}.png", dpi=150, bbox_inches="tight")
    plt.close()

    y_test_orig = np.expm1(y_test.values)
    n_plot = min(300, len(y_test))

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(range(n_plot), y_test_orig[:n_plot], "k-", lw=1.4, label="Actual", zorder=10)
    for i, (name, yp_log) in enumerate(all_preds.items()):
        ax.plot(range(n_plot), np.expm1(yp_log)[:n_plot], lw=0.8, alpha=0.6,
                color=PALETTE[i % len(PALETTE)], label=name)
    ax.set_xlabel("Test sample index"); ax.set_ylabel("Review count")
    ax.set_title(f"All models — {RUN_LABEL}", fontweight="bold")
    ax.legend(fontsize=8, ncol=3, loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/predictions_vs_actual_{RUN_LABEL}.png", dpi=150, bbox_inches="tight")
    plt.close()

    best_name = results_df.iloc[0]["Model"]
    y_best = np.expm1(all_preds[best_name])
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(n_plot), y_test_orig[:n_plot], "k-", lw=1.2, label="Actual", zorder=5)
    ax.plot(range(n_plot), y_best[:n_plot], "-", lw=0.9, alpha=0.8,
            color="#E15759", label=f"{best_name} (best)")
    ax.set_xlabel("Test sample index"); ax.set_ylabel("Review count")
    ax.set_title(f"Best model: {best_name} — {RUN_LABEL}", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/predictions_best_{RUN_LABEL}.png", dpi=150, bbox_inches="tight")
    plt.close()

    importances = pd.Series(rf.feature_importances_, index=X_train.columns)
    top20 = importances.nlargest(20).sort_values()
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ["#4E79A7" if f in CORE_IVS else "#BAB0AC" for f in top20.index]
    top20.plot(kind="barh", ax=ax, color=colors, alpha=0.85)
    ax.legend(handles=[mpatches.Patch(facecolor="#4E79A7", label="Core IV"),
                       mpatches.Patch(facecolor="#BAB0AC", label="Control")],
              loc="lower right", fontsize=9)
    ax.set_xlabel("Feature Importance (MDI)")
    ax.set_title(f"Top 20 Feature Importances — {RUN_LABEL}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/feature_importance_{RUN_LABEL}.png", dpi=150, bbox_inches="tight")
    plt.close()

    best = results_df.iloc[0]
    master_summary.append({
        "experiment": RUN_LABEL,
        "poi_radius": exp["poi"],
        "n_core_IVs": len(CORE_IVS),
        "n_features": len(ALL_FEATURES),
        "best_model": best["Model"],
        "MAE":       round(best["MAE"], 2),
        "RMSE":      round(best["RMSE"], 2),
        "RMSE_log":  round(best["RMSE_log"], 4),
        "MAPE":      round(best["MAPE"], 2),
        "R2":        round(best["R2"], 4),
        "Adj_R2":    round(best["Adj_R2"], 4),
        "R2_log":    round(best["R2_log"], 4),
    })


  [1/3] M_5_POI500m   POI=Total_POI_500m
  After 95-percentile trim: 4927 rows
  Tuning Elastic Net ...
  Tuning Random Forest ...
  Tuning XGBoost (300 trials) ...
  Tuning LightGBM (200 trials) ...
  Fitting all models ...

  Best: XGBoost (plain)  RMSE_log=0.7841  R²=0.2872

  [2/3] M_6_POI800m   POI=Total_POI_800m
  After 95-percentile trim: 4927 rows
  Tuning Elastic Net ...
  Tuning Random Forest ...
  Tuning XGBoost (300 trials) ...
  Tuning LightGBM (200 trials) ...
  Fitting all models ...

  Best: LightGBM  RMSE_log=0.7887  R²=0.2884

  [3/3] M_7_POI1500m   POI=Total_POI_1500m
  After 95-percentile trim: 4927 rows
  Tuning Elastic Net ...
  Tuning Random Forest ...
  Tuning XGBoost (300 trials) ...
  Tuning LightGBM (200 trials) ...
  Fitting all models ...

  Best: LightGBM  RMSE_log=0.7854  R²=0.2853


In [9]:
#Save master summary

summary_df = pd.DataFrame(master_summary)
summary_df.to_csv(f"{OUTPUT_DIR}/master_summary_rc_script2.csv", index=False)

print("\n" + "=" * 90)
print(f"  MASTER SUMMARY — Script 2 (review_count, {len(EXPERIMENTS)} experiments)")
print("=" * 90)
print(summary_df.to_string(index=False))
print(f"\n  Saved: {OUTPUT_DIR}/master_summary_rc_script2.csv")


  MASTER SUMMARY — Script 2 (review_count, 3 experiments)
  experiment      poi_radius  n_core_IVs  n_features      best_model   MAE  RMSE  RMSE_log  MAPE     R2  Adj_R2  R2_log
 M_5_POI500m  Total_POI_500m           1          48 XGBoost (plain) 42.95 62.46    0.7841 85.81 0.2872  0.2424  0.4447
 M_6_POI800m  Total_POI_800m           1          48        LightGBM 42.40 62.40    0.7887 86.07 0.2884  0.2437  0.4380
M_7_POI1500m Total_POI_1500m           1          48        LightGBM 42.77 62.54    0.7854 86.42 0.2853  0.2404  0.4428

  Saved: ./results_rc_script2/master_summary_rc_script2.csv
